# GETTING STARTED WITH LANGCHAIN
- Simple LLM calls with streaming
- Dynamic prompt templates (translation app)
- Building chains (story generator with analysis)
- Conversational QnA assistant with memory
- Tool integration (calculator and weather)

In [1]:
import langchain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
os.environ["GOOGLE_GEMINI_API_KEY"] = os.getenv("GOOGLE_GEMINI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Example 1: Simple LLM Call With Streaming

In [8]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

In [71]:
model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7f0ac80e48d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7f0ac80e4d10>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [72]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="groq:openai/gpt-oss-120b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.3'}}, client=<groq.resources.chat.completions.Completions object at 0x7f0ac80e56a0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7f0ac80e5f20>, model_name='groq:openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [73]:
## Create messages  
messages = [
    SystemMessage("You are a helpful AI assistant, give a long essay on the topic."),
    HumanMessage("What are the top 2 benefits of using LangChain?") 
] 

In [74]:
## Invoke the model
response = model.invoke(messages)
response

AIMessage(content='**Top\u202f2 Benefits of Using LangChain**\n\n| Benefit | Why It Matters | How It Helps You |\n|---------|----------------|------------------|\n| **1️⃣ Rapid Development of LLM‑Powered Applications** | LangChain abstracts away the low‑level plumbing (prompt templating, model I/O, memory handling, tool integration, etc.) so you can focus on the core logic of your product. | • **Composable building blocks** – chains, agents, memory, callbacks, and utilities can be mixed‑and‑matched with just a few lines of code.<br>• **Unified API** across many providers (OpenAI, Anthropic, Cohere, Azure, etc.) and model types (chat, embeddings, embeddings‑as‑a‑service).<br>• **Extensive examples & templates** – starter projects for chatbots, retrieval‑augmented generation (RAG), autonomous agents, data extraction, and more, dramatically cutting prototype time. |\n| **2️⃣ Scalable & Extensible Architecture for Complex Workflows** | Real‑world LLM solutions often require chaining multip

In [75]:
print(response.content)

**Top 2 Benefits of Using LangChain**

| Benefit | Why It Matters | How It Helps You |
|---------|----------------|------------------|
| **1️⃣ Rapid Development of LLM‑Powered Applications** | LangChain abstracts away the low‑level plumbing (prompt templating, model I/O, memory handling, tool integration, etc.) so you can focus on the core logic of your product. | • **Composable building blocks** – chains, agents, memory, callbacks, and utilities can be mixed‑and‑matched with just a few lines of code.<br>• **Unified API** across many providers (OpenAI, Anthropic, Cohere, Azure, etc.) and model types (chat, embeddings, embeddings‑as‑a‑service).<br>• **Extensive examples & templates** – starter projects for chatbots, retrieval‑augmented generation (RAG), autonomous agents, data extraction, and more, dramatically cutting prototype time. |
| **2️⃣ Scalable & Extensible Architecture for Complex Workflows** | Real‑world LLM solutions often require chaining multiple steps (retrieval, reasonin

In [76]:
## Streaming example
for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

**Top 2 Benefits of Using LangChain**

| # | Benefit | Why It Matters |
|---|---------|----------------|
| **1️⃣ Accelerated Development of LLM‑Powered Applications** | LangChain supplies a **modular, plug‑and‑play framework** that stitches together large language models (LLMs), data sources, memory stores, and tooling. Instead of writing low‑level integration code for every new use‑case, developers can assemble pipelines from pre‑built components (e.g., prompt templates, retrievers, agents, callbacks). This dramatically reduces the time‑to‑prototype and lets teams focus on domain logic rather than boilerplate. |
| **2️⃣ Scalable, Maintainable Architecture for Complex Workflows** | LangChain encourages a **clear separation of concerns** (prompt management, chaining, routing, caching, and state handling). By abstracting each step into reusable objects, the same codebase can scale from a simple “question‑answer” bot to multi‑step reasoning agents, retrieval‑augmented generation (RAG) pip

### Dynamic Templates

In [77]:
from langchain_core.prompts import ChatPromptTemplate

## create translation app

translation_template = ChatPromptTemplate.from_messages([
    ("system", "You are a professional translator. Translate the following {text} from {source_language} to {target_language}. Maintain the tone and style."),
    ("user", "{text}")
])

## using the template
prompt = translation_template.invoke({
    "source_language": "English",
    "target_language": "Turkish",
    "text": "langchain makes building AI applications incredibly easy!"
})

In [78]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator. Translate the following langchain makes building AI applications incredibly easy! from English to Turkish. Maintain the tone and style.', additional_kwargs={}, response_metadata={}), HumanMessage(content='langchain makes building AI applications incredibly easy!', additional_kwargs={}, response_metadata={})])

In [79]:
for chunk in model.stream(prompt):
    print(chunk.content, end="", flush=True)

langchain, AI uygulamaları oluşturmayı inanılmaz derecede kolaylaştırıyor!

### Building Your First Chain

In [80]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda 

def create_story_chain():
    ## Template for story generation
    story_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a creative storyteller. Write a short and engaging story based on a given {theme}"),
        ("user", "Theme: {theme}\n Main Character: {character}\n Setting: {setting}"),
    ])

    ## Template for story analysis
    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a literary critic. Analyze the following story and provide insights."),
        ("user", "{story}")   
    ])

    story_chain = (
        story_prompt| model| StrOutputParser()
    )

    def analyze_story(story_text):
        return {"story": story_text}

    analysis_chain = (
        story_chain | RunnableLambda(analyze_story) | analysis_prompt | model | StrOutputParser()
    )

    return analysis_chain

In [81]:
chain = create_story_chain()
chain

ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['theme'], input_types={}, partial_variables={}, template='You are a creative storyteller. Write a short and engaging story based on a given {theme}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme: {theme}\n Main Character: {character}\n Setting: {setting}'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audi

In [83]:
for chunk in chain.stream({
    "theme": "Friendship and Adventure",
    "character": "A young explorer named Alex",
    "setting": "A mysterious island filled with hidden treasures"
}):
    print(chunk, end="", flush=True)

# result = chain.invoke({
#     "theme": "Friendship and Adventure",
#     "character": "A young explorer named Alex",
#     "setting": "A mysterious island filled with hidden treasures"
# })

# print("Story and Analysis:\n", result)

## Literary Analysis of **“The Secret of Whispering Cove”**

### 1. Overview & Narrative Arc  

| Element | Description | Effect |
|---|---|---|
| **Protagonist** | Alex – a classic “seeker” driven by family‑borne legends. | Functions as the reader’s surrogate; his curiosity propels the plot. |
| **Companion** | Kiko the parrot – witty, talkative, and magically attuned to the island. | Serves as a guide‑mentor, comic relief, and the story’s “other half.” |
| **Structure** | Three‑stage quest (Labyrinth → Bridge → Chamber) mirroring the Hero’s Journey (Departure → Initiation → Return). | Provides a clear, escalating tension and a satisfying rhythm of discovery. |
| **Resolution** | The “treasure” is a journal + compass that celebrate friendship rather than material wealth. | Subverts the conventional treasure‑hunt climax, reinforcing the story’s moral. |

The tale follows a tight, almost formulaic quest pattern, but it injects fresh emotional resonance through the partnership between a 

## Trying Gemini Model

In [9]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

In [4]:
model = init_chat_model("google_genai:gemini-3.7-flash")

In [13]:
messages = [
    SystemMessage("You are a helpful AI assistant, give a long essay on the topic. Think good before generating"),
    HumanMessage("What are the top 2 benefits of using LangChain?")
]

In [16]:
# for chunk in model.stream(messages):
#     for block in chunk.content_blocks:
#         if block["type"] == "reasoning" and (reasoning := block.get("reasoning")):
#             print(f"Reasoning: {reasoning}")
#         elif block["type"] == "tool_call_chunk":
#             print(f"Tool call chunk: {block}")
#         elif block["type"] == "text":
#             print(block["text"])
#         else:
#             print(f"Unknown block type: {block}")

for chunk in model.stream(messages):
    reasoning_steps = [r for r in chunk.content_blocks if r["type"] == "reasoning"]
    print(reasoning_steps if reasoning_steps else chunk.text)

# The Dual Pillars of Modern AI Development: Analyzing the Top Two Benefits of LangChain

The advent of Large Language Models (LL
Ms) like OpenAI’s GPT-4, Anthropic’s Claude, and open-weight models like Meta’s Llama has
 fundamentally transformed software engineering. However, building enterprise-grade, production-ready applications with raw LLMs quickly reveals their intrinsic
 limitations: they are stateless, prone to hallucinations, bounded by their training cutoff dates, and isolated from external software ecosystems. 


To bridge the gap between raw statistical text generators and intelligent, production-ready software, **LangChain** has emerged as the
 definitive orchestration framework. While LangChain offers a vast array of utilities—ranging from output parsing to evaluation suites—its transformative
 value fundamentally rests on two overarching pillars:

1. **Modular Abstraction and Model Agnosticism (Architectural Flexibility
 and Future-Proofing)**
2. **Seamless Context-Augm

GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}